# Model Tuning

This notebook tunes Random Forest and Gradient Boosting models using only the 2021-2024 training period. The 2025 holdout set is reserved for final testing. Models are selected by cross-validated RMSE.

### Objectives:

1. Load processed training data.
2. Build chronological CV folds.
3. Run `GridSearchCV` for Random Forest and Gradient Boosting.
4. Save the best parameters and fitted model artifacts.

In [1]:
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
from sklearn.ensemble import GradientBoostingRegressor, RandomForestRegressor
from sklearn.model_selection import GridSearchCV

In [3]:
repo = Path.cwd().parent
# If the notebook is running from /notebooks, move one level up to the project root
if not (repo / "data").exists():
    repo = repo.parent

processed_dir = repo / "data" / "processed_salary_fix"
result_dir = repo / "results" / "Fixed"
art_dir = repo / "artifacts" / "Fixed"

result_dir.mkdir(parents=True, exist_ok=True)
art_dir.mkdir(parents=True, exist_ok=True)

x_train = pd.read_csv(processed_dir / "X_train_processed.csv")
y_train = pd.read_csv(processed_dir / "y_train.csv")["salary"]
lookup_train = pd.read_csv(processed_dir / "player_lookup_train.csv")

print(x_train.shape, y_train.shape)
print(lookup_train.groupby("year").size())

(633, 26) (633,)
year
2021    155
2022    166
2023    155
2024    157
dtype: int64


In [4]:
# Prepare chronological CV folds for GridSearchCV
cv_folds = []
years = sorted(lookup_train["year"].unique())

# Each fold trains on past seasons and validates on the next season
for val_year in years[1:]:
    train_years = [year for year in years if year < val_year]
    train_idx = lookup_train.index[lookup_train["year"].isin(train_years)].to_numpy()
    val_idx = lookup_train.index[lookup_train["year"] == val_year].to_numpy()
    cv_folds.append((train_idx, val_idx))

for i, (train_idx, val_idx) in enumerate(cv_folds, start=1):
    train_years = [int(year) for year in sorted(lookup_train.loc[train_idx, "year"].unique())]
    val_years = [int(year) for year in sorted(lookup_train.loc[val_idx, "year"].unique())]

    train_label = ",".join(map(str, train_years))
    val_label = "-".join(map(str, val_years))

    print(f"fold {i}: train {train_label} ({len(train_idx)} rows) -> val {val_label} ({len(val_idx)} rows)")

fold 1: train 2021 (155 rows) -> val 2022 (166 rows)
fold 2: train 2021,2022 (321 rows) -> val 2023 (155 rows)
fold 3: train 2021,2022,2023 (476 rows) -> val 2024 (157 rows)


### Hyperparameter Search Design

`modeling_experiments.ipynb` showed that the Stacking Ensemble and ElasticNet had strong CV RMSE values, but the Stacking Ensemble is less interpretable and more complex because it combines several models, which can be risky with a small dataset. ElasticNet performed well, but it is part of the regularized linear model family already represented by Ridge. For this optimization step, we focused on RandomForest and Gradient Boosting because they are defensible tree-based ensemble models that can capture nonlinear relationships while keeping the tuning workflow simple and reproducible.


1. The search space is centered around the baseline model settings.
2. Because the dataset is relatively small, we avoid an overly large grid that could overfit the CV folds.
3. We tune only the main complexity-control parameters for Random Forest and Gradient Boosting.
4. We limit tuning to two model families to avoid a “soup model” approach and keep model selection defensible.

In [6]:
# RandomForestRegressor hyperparameter
RandomForest_grid = {
    "n_estimators": [200, 400],
    "max_depth": [None, 6, 10],
    "min_samples_leaf": [3, 5, 8],
}

# GradientBoostingRegressor hyperparameter
GradientBoosting_grid = {
    "n_estimators": [100, 200],
    "learning_rate": [0.03, 0.05, 0.1],
    "max_depth": [2, 3],
    "min_samples_leaf": [3, 5],
}

searches = {
    "RandomForest": (
        RandomForestRegressor(random_state=26),
        RandomForest_grid,
        art_dir / "best_RandomForest_fixed.joblib",
    ),
    "GradientBoosting": (
        GradientBoostingRegressor(random_state=26),
        GradientBoosting_grid,
        art_dir / "best_GradientBoosting_fixed.joblib",
    ),
}

In [7]:
# Initialize containers for grid-search results and best model tracking
rows = []
best_rows = []
best_score = np.inf
best_name = None
best_model = None

# Run GridSearchCV for each model family defined in searches
for name, (model, grid, path) in searches.items():
    search = GridSearchCV(
        estimator=model,
        param_grid=grid,
        scoring="neg_root_mean_squared_error",  # RMSE, negative for sklearn scoring
        cv=cv_folds,
        refit=True,
        n_jobs=-1,
        return_train_score=True,
    )
    search.fit(x_train, y_train)

    # Convert the full GridSearchCV output into a cleaner results table
    result = pd.DataFrame(search.cv_results_)
    keep_cols = [
        "params",
        "mean_test_score",  # Mean validation score across folds
        "std_test_score",   # Fold-to-fold validation score variation
        "rank_test_score",  # Rank of each parameter setting
        "mean_train_score", # Mean training score across folds
    ]
    result = result[keep_cols]
    result["model"] = name
    result["cv_rmse"] = -result["mean_test_score"]
    result["train_rmse"] = -result["mean_train_score"]
    result = result.drop(columns=["mean_test_score", "mean_train_score"])
    rows.append(result)

    # Store the best CV result for this model family
    cv_rmse = -search.best_score_
    best_row = {
        "model": name,
        "cv_rmse": cv_rmse,
        "params": search.best_params_,
        "artifact": str(path.relative_to(repo)),
    }
    best_rows.append(best_row)

    # Save this model family's best estimator as a reusable artifact
    joblib.dump(search.best_estimator_, path)

    if cv_rmse < best_score:
        best_score = cv_rmse
        best_name = name
        best_model = search.best_estimator_

all_rows = pd.concat(rows, ignore_index=True)
best_df = pd.DataFrame(best_rows).sort_values("cv_rmse")

all_rows.to_csv(result_dir / "tuning_results_fixed.csv", index=False)
joblib.dump(best_model, art_dir / "best_model_fixed.joblib")

print("best model:", best_name)
display(best_df)

best model: GradientBoosting


,model,cv_rmse,params,artifact
1,GradientBoosting,39765.109349,"{'learning_rate': 0.05, 'max_depth': 2, 'min_s...",artifacts\Fixed\best_GradientBoosting_fixed.jo...
0,RandomForest,40118.980248,"{'max_depth': 6, 'min_samples_leaf': 5, 'n_est...",artifacts\Fixed\best_RandomForest_fixed.joblib


**Double-check if we use `processed_salary_fix` and new features created by `train_test_proocessed_fixed.ipynb1`.**

In [8]:
print(processed_dir)

x_train = pd.read_csv(processed_dir / "X_train_processed.csv")
feature_names = pd.read_csv(processed_dir / "feature_names.csv")["feature"].tolist()

print(x_train.shape)
print(len(feature_names))
print(feature_names)

c:\Users\lelin\Documents\GitHub\summer26-wnba-player-valuation\data\processed_salary_fix
(633, 26)
26
['avail_rate', 'blk', 'fg', 'fg_per_g', 'fga', 'ft', 'fta', 'g', 'mp', 'pca1', 'pts', 'pts_rookie', 'pts_vet', 'pts_hardship', 'start_rate', 'team_min', 'tov', 'ws_rookie', 'ws_vet', 'ws_hardship', 'group_controlled', 'group_hardship', 'group_other', 'group_rookie', 'group_unknown', 'group_veteran']
